# 🍓 Strawberry Classification Project Manager

Welcome to your YOLO11 strawberry classification project! This notebook provides individual executable cells for each script in your workflow.

## 📋 Features
- **Simple Execution**: Each script has its own cell that you can run directly
- **Configurable Parameters**: Modify variables in each cell to customize script arguments
- **Clear Workflow**: Organized sections for data preparation, training, validation, and prediction
- **No Complex UI**: Just straightforward Python cells you can execute

## 🚀 Getting Started
1. Run the setup cells below to configure your environment
2. Modify the variables in each script cell as needed
3. Execute the cells in order for your workflow

**Note**: Make sure to run the setup cells first to ensure all paths and configurations are ready!

## 📚 Setup - Import Libraries & Configure Environment

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime

# Configure paths
PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
DATA_DIR = PROJECT_ROOT / "data"
UTILS_DIR = PROJECT_ROOT / "utils"
MODELS_DIR = PROJECT_ROOT / "models"
VISUALIZATIONS_DIR = PROJECT_ROOT / "visualizations"

# Ensure directories exist
for dir_path in [SCRIPTS_DIR, DATA_DIR, UTILS_DIR, MODELS_DIR, VISUALIZATIONS_DIR]:
    dir_path.mkdir(exist_ok=True)

# Add project directories to Python path
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(UTILS_DIR))

print("✅ Environment setup complete!")
print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📜 Scripts directory: {SCRIPTS_DIR}")
print(f"🐍 Python executable: {sys.executable}")

## 📊 Data Preparation

Use this section to prepare your raw datasets for training. The script will split your data into train/val/test sets and create necessary configuration files.

In [ ]:
# Configuration for data preparation
DATASET_NAME = "farm_dataset"  # Change this to your dataset name
TRAIN_RATIO = 0.7
VAL_RATIO = 0.2
TEST_RATIO = 0.1
SEED = 42

# Execute data preparation (this will regenerate the config with correct paths)
import subprocess
import sys

print(f"🔄 Preparing dataset: {DATASET_NAME}")
print(f"📊 Split ratios - Train: {TRAIN_RATIO}, Val: {VAL_RATIO}, Test: {TEST_RATIO}")
print("📁 This will regenerate the dataset config with correct paths")
print("=" * 60)

try:
    cmd = [
        sys.executable, 
        str(SCRIPTS_DIR / "prepare_data.py"),
        "--source-dir", f"{str(DATA_DIR)}/datasets/{DATASET_NAME}",
        "--dataset-name", DATASET_NAME,
        "--train-ratio", str(TRAIN_RATIO),
        "--val-ratio", str(VAL_RATIO),
        "--test-ratio", str(TEST_RATIO),
        "--seed", str(SEED)
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    
    if result.returncode == 0:
        print("✅ Data preparation completed successfully!")
        print(result.stdout)
        
        # Show the regenerated config file
        config_file = DATA_DIR / "processed" / DATASET_NAME / "dataset_config.yaml"
        if config_file.exists():
            print(f"\n📄 Generated config file at: {config_file}")
            with open(config_file, 'r') as f:
                print("Content:")
                print(f.read())
    else:
        print("❌ Error during data preparation:")
        print(result.stderr)
        
except Exception as e:
    print(f"❌ Failed to run data preparation: {e}")

## 📈 Dataset Analysis

Analyze your dataset to understand the distribution of classes and generate visualizations.

## 🎨 Data Augmentation Setup

Setup color cast augmentation to be applied during training (not pre-processing). This creates an augmentation configuration that will be used during the training process for on-the-fly augmentation.

In [ ]:
# Configuration for dataset analysis
DATASET_NAME = "farm_dataset"  # Change this to your dataset name

# Execute dataset analysis
print(f"📊 Analyzing dataset: {DATASET_NAME}")
print("=" * 60)

try:
    cmd = [
        sys.executable, 
        str(SCRIPTS_DIR / "analyze_dataset.py"),
        "--dataset-path", str(DATA_DIR / "processed" / DATASET_NAME),
        "--dataset-name", DATASET_NAME
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    
    if result.returncode == 0:
        print("✅ Dataset analysis completed successfully!")
        print(result.stdout)
        
        # Display generated plots
        from IPython.display import Image, display
        import matplotlib.pyplot as plt
        import matplotlib.image as mpimg
        
        print("\n📈 Generated Visualizations:")
        print("=" * 40)
        
        # Look for generated plot files
        plot_files = list(VISUALIZATIONS_DIR.glob(f"*{DATASET_NAME}*.png"))
        
        if plot_files:
            for plot_file in sorted(plot_files):
                print(f"\n📊 {plot_file.name}")
                try:
                    # Display the plot
                    img = mpimg.imread(str(plot_file))
                    plt.figure(figsize=(10, 6))
                    plt.imshow(img)
                    plt.axis('off')
                    plt.title(plot_file.stem.replace('_', ' ').title())
                    plt.tight_layout()
                    plt.show()
                except Exception as e:
                    print(f"   ⚠️ Could not display {plot_file.name}: {e}")
        else:
            print("   ℹ️ No visualization files found in the visualizations directory")
            
    else:
        print("❌ Error during dataset analysis:")
        print(result.stderr)
        
except Exception as e:
    print(f"❌ Failed to run dataset analysis: {e}")

## 🎨 Training with Custom Augmentation

Train your YOLO11 model with custom color cast augmentation applied on-the-fly during training. This approach is more efficient than pre-processing the entire dataset.

In [ ]:
# Configuration for training with custom augmentation
DATASET_NAME = "farm_dataset"  # Change this to your dataset name
REFERENCE_IMAGE = "scripts/reference_blue_light.jpg"  # Path to reference image with desired lighting
MODEL_SIZE = "yolo11l.pt"  # Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt
EPOCHS = 200
BATCH_SIZE = 8   # Reduced for RTX 4070 memory constraints
IMAGE_SIZE = 640  # Image size for training
DEVICE = "0"  # auto, cpu, or cuda:0
PROJECT_NAME = "strawberry_classifier_custom_aug"  # Experiment name

# Color cast augmentation settings
COLOR_CAST_PROBABILITY = 0.5  # Probability of applying color cast during training
COLOR_CAST_INTENSITY = 1.0  # Intensity of color cast effect

# TensorBoard Configuration
ENABLE_TENSORBOARD = True  # Set to True to enable TensorBoard logging

print("🎨 Training with Custom Color Cast Augmentation:")
print(f"   📂 Dataset: {DATASET_NAME}")
print(f"   📸 Reference image: {REFERENCE_IMAGE}")
print(f"   📦 Model: {MODEL_SIZE}")
print(f"   🔄 Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")
print(f"   🖼️ Image size: {IMAGE_SIZE}, Device: {DEVICE}")
print(f"   🎲 Color cast probability: {COLOR_CAST_PROBABILITY}")
print(f"   💪 Color cast intensity: {COLOR_CAST_INTENSITY}")
print(f"   📈 TensorBoard: {'ENABLED' if ENABLE_TENSORBOARD else 'DISABLED'}")
print()

# Check if reference image exists
reference_path = PROJECT_ROOT / REFERENCE_IMAGE
if not reference_path.exists():
    print(f"❌ Reference image not found: {reference_path}")
    print("   Please run the 'Data Augmentation Setup' section first")
else:
    print(f"✅ Reference image found: {reference_path}")
    
    # Execute training with custom augmentation
    print(f"🚀 Starting training with custom color cast augmentation...")
    print("=" * 70)
    
    try:
        data_config = str(DATA_DIR / "processed" / DATASET_NAME / "data.yaml")
        
        cmd = [
            sys.executable, 
            str(SCRIPTS_DIR / "train_with_augmentation.py"),
            "--data", data_config,
            "--reference-image", str(reference_path),
            "--model", MODEL_SIZE,
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--img-size", str(IMAGE_SIZE),
            "--device", DEVICE,
            "--name", PROJECT_NAME,
            "--color-cast-prob", str(COLOR_CAST_PROBABILITY),
            "--color-cast-intensity", str(COLOR_CAST_INTENSITY)
        ]
        
        # Add TensorBoard flag if enabled
        if ENABLE_TENSORBOARD:
            cmd.append("--tensorboard")
        
        print("🎨 Custom color cast augmentation will be applied during training")
        print("📊 YOLO's default color augmentations are disabled to avoid conflicts")
        print("🔄 Geometric augmentations (flip, rotate, etc.) remain active")
        print("=" * 70)
        
        # Run with real-time output
        result = subprocess.run(cmd, cwd=PROJECT_ROOT)
        
        if result.returncode == 0:
            print("\n" + "=" * 70)
            print("✅ Training with custom augmentation completed successfully!")
            
            # Show where results are saved
            results_dir = PROJECT_ROOT / "runs" / "train" / PROJECT_NAME
            print(f"📁 Results saved to: {results_dir}")
            
            # List key output files
            if results_dir.exists():
                print(f"📊 Key files generated:")
                key_files = ["weights/best.pt", "weights/last.pt", "results.csv"]
                for file_name in key_files:
                    file_path = results_dir / file_name
                    if file_path.exists():
                        print(f"  ✅ {file_name}")
                    else:
                        print(f"  ⚠️ {file_name} (not found)")
                        
                print(f"\n🎨 Training used custom color cast augmentation from: {reference_path}")
                print(f"💡 Compare results with standard training to see the impact!")
                
        else:
            print(f"\n❌ Training failed with exit code: {result.returncode}")
            
    except Exception as e:
        print(f"❌ Failed to run training with custom augmentation: {e}")

print(f"\n💡 Benefits of this approach:")
print("• 🚀 More efficient - no need to pre-process entire dataset")
print("• 🎯 On-the-fly augmentation during training")
print("• 🔄 Each epoch sees different augmented versions")
print("• 💾 Saves disk space - no duplicate augmented images")
print("• 🎨 Precise color cast matching from reference image")

## 🎨 Default Training

In [ ]:
# Configuration for model training
DATASET_NAME = "farm_dataset"  # Change this to your dataset name
MODEL = "runs/train/open_source_dataset_640_l/weights/best.pt"  # Options: yolo11n.pt, yolo11s.pt, yolo11m.pt, yolo11l.pt, yolo11x.pt
EPOCHS = 200
PATIENCE = 20
BATCH_SIZE = 16
IMAGE_SIZE = 640
DEVICE = "0"  # auto, cpu, or cuda:0
PROJECT_NAME = "farm_dataset_640_l_pretrained_frozen_17"  # Experiment name

# TensorBoard Configuration
ENABLE_TENSORBOARD = True  # Set to True to enable TensorBoard logging

# Execute model training with real-time progress
print(f"🤖 Training YOLO11 model on dataset: {DATASET_NAME}")
print(f"📊 Model: {MODEL}, Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")
print(f"🖼️  Image size: {IMAGE_SIZE}, Device: {DEVICE}")
if ENABLE_TENSORBOARD:
    print("📈 TensorBoard logging: ENABLED")
    print("   💡 After training starts, run the TensorBoard monitoring cell below to view real-time metrics!")
else:
    print("📈 TensorBoard logging: DISABLED")
print("=" * 60)

try:
    # Path to the dataset config file (use standardized data.yaml)
    data_config = str(DATA_DIR / "processed" / DATASET_NAME / "data.yaml")
    
    cmd = [
        sys.executable, 
        str(SCRIPTS_DIR / "train.py"),
        "--data", data_config,
        "--model", MODEL,
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--img-size", str(IMAGE_SIZE),
        "--patience", str(PATIENCE),
        "--device", DEVICE,
        "--name", PROJECT_NAME
    ]
    
    # Add TensorBoard flag if enabled
    if ENABLE_TENSORBOARD:
        cmd.append("--tensorboard")
    
    print("🚀 Starting training with live progress...")
    print("📊 You should see epoch progress, loss values, and metrics below:")
    if ENABLE_TENSORBOARD:
        print("📈 TensorBoard logs will be saved to runs/train/{}/".format(PROJECT_NAME))
    print("=" * 60)
    
    # Run with real-time output (no capture_output=True)
    result = subprocess.run(cmd, cwd=PROJECT_ROOT)
    
    if result.returncode == 0:
        print("\n" + "=" * 60)
        print("✅ Model training completed successfully!")
        
        # Show where results are saved
        results_dir = PROJECT_ROOT / "runs" / "train" / PROJECT_NAME
        print(f"📁 Results saved to: {results_dir}")
        
        # List key output files
        if results_dir.exists():
            print(f"📊 Key files generated:")
            key_files = ["weights/best.pt", "weights/last.pt", "results.csv", "confusion_matrix.png"]
            for file_name in key_files:
                file_path = results_dir / file_name
                if file_path.exists():
                    print(f"  ✅ {file_name}")
                else:
                    print(f"  ⚠️  {file_name} (not found)")
                    
            # Check for TensorBoard logs
            if ENABLE_TENSORBOARD:
                tb_dir = results_dir / "events.out.tfevents*"
                tb_files = list(results_dir.glob("events.out.tfevents*"))
                if tb_files:
                    print(f"  ✅ TensorBoard logs (found {len(tb_files)} files)")
                else:
                    print(f"  ⚠️  TensorBoard logs (not found)")
    else:
        print(f"\n❌ Training failed with exit code: {result.returncode}")
        
except Exception as e:
    print(f"❌ Failed to run model training: {e}")

## 🎯 Training a Classifier

Train a YOLO11 classification model using the extracted crops from your detection dataset. This creates a classifier that predicts the class of individual objects.

**Classification vs Detection:**
- **Classification**: Predicts what class the entire image belongs to (e.g., "strawberry", "flower")
- **Detection**: Finds objects in images and draws bounding boxes around them

**Requirements:**
- Run the "Extract Crops for Classification" section first
- Classification dataset structure: `data/classification/{dataset_name}/train/class1/`, etc.
- Detection models are automatically converted to classification models

In [ ]:
# Configuration for classification model training
DATASET_NAME = "farm_dataset"  # Change this to your dataset name
MODEL = "yolo11l-cls.pt"
EPOCHS = 200
PATIENCE = 20
BATCH_SIZE = 16
IMAGE_SIZE = 640
DEVICE = "0"  # auto, cpu, or cuda:0
PROJECT_NAME = "farm_dataset_classifier"  # Experiment name

# TensorBoard Configuration
ENABLE_TENSORBOARD = True  # Set to True to enable TensorBoard logging

# Execute classification model training with real-time progress
print(f"🎯 Training YOLO11 classifier on dataset: {DATASET_NAME}")
print(f"📊 Model: {MODEL} (will be converted to classification), Epochs: {EPOCHS}, Batch size: {BATCH_SIZE}")
print(f"🖼️  Image size: {IMAGE_SIZE}, Device: {DEVICE}")
print(f"📁 Using classification dataset structure (train/val/test folders with class subdirectories)")
if ENABLE_TENSORBOARD:
    print("📈 TensorBoard logging: ENABLED")
    print("   💡 After training starts, run the TensorBoard monitoring cell below to view real-time metrics!")
else:
    print("📈 TensorBoard logging: DISABLED")
print("=" * 60)

try:
    # Path to the classification dataset directory (not data.yaml)
    classification_data_path = str(DATA_DIR / "classification" / DATASET_NAME)
    
    # Check if classification dataset exists
    if not Path(classification_data_path).exists():
        print(f"❌ Classification dataset not found at: {classification_data_path}")
        print("💡 You need to create a classification dataset first.")
        print("   Run the 'Extract Crops for Classification' section to generate it from detection data.")
    else:
        print(f"✅ Classification dataset found at: {classification_data_path}")
        
        cmd = [
            sys.executable, 
            str(SCRIPTS_DIR / "train.py"),
            "--task", "classify",  # Use classification mode
            "--data", classification_data_path,
            "--model", MODEL,
            "--epochs", str(EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--img-size", str(IMAGE_SIZE),
            "--patience", str(PATIENCE),
            "--device", DEVICE,
            "--name", PROJECT_NAME
        ]
        
        # Add TensorBoard flag if enabled
        if ENABLE_TENSORBOARD:
            cmd.append("--tensorboard")
        
        print("🚀 Starting classification training with live progress...")
        print("🔄 Detection model will be automatically converted to classification model")
        print("📊 You should see epoch progress, loss values, and accuracy metrics below:")
        if ENABLE_TENSORBOARD:
            print("📈 TensorBoard logs will be saved to runs/train/{}/".format(PROJECT_NAME))
        print("=" * 60)
        
        # Run with real-time output (no capture_output=True)
        result = subprocess.run(cmd, cwd=PROJECT_ROOT)
        
        if result.returncode == 0:
            print("\n" + "=" * 60)
            print("✅ Classification model training completed successfully!")
            
            # Show where results are saved
            results_dir = PROJECT_ROOT / "runs" / "train" / PROJECT_NAME
            print(f"📁 Results saved to: {results_dir}")
            
            # List key output files for classification
            if results_dir.exists():
                print(f"📊 Key files generated:")
                key_files = ["weights/best.pt", "weights/last.pt", "results.csv", "confusion_matrix.png", "confusion_matrix_normalized.png"]
                for file_name in key_files:
                    file_path = results_dir / file_name
                    if file_path.exists():
                        print(f"  ✅ {file_name}")
                    else:
                        print(f"  ⚠️  {file_name} (not found)")
                        
                # Check for TensorBoard logs
                if ENABLE_TENSORBOARD:
                    tb_files = list(results_dir.glob("events.out.tfevents*"))
                    if tb_files:
                        print(f"  ✅ TensorBoard logs (found {len(tb_files)} files)")
                    else:
                        print(f"  ⚠️  TensorBoard logs (not found)")
                        
                print(f"\n🎯 Classification model metrics:")
                print(f"   • Top-1 accuracy")
                print(f"   • Top-5 accuracy") 
                print(f"   • Per-class confusion matrix")
                print(f"   • Training/validation loss curves")
        else:
            print(f"\n❌ Classification training failed with exit code: {result.returncode}")
            
except Exception as e:
    print(f"❌ Failed to run classification model training: {e}")

print(f"\n💡 Classification vs Detection:")
print("🎯 Classification: Predicts what class the entire image belongs to")
print("📦 Detection: Finds objects in images and draws bounding boxes")
print("📁 Classification uses directory structure: train/class1/, train/class2/, etc.")
print("📄 Detection uses data.yaml with image/label pairs")

🎯 Training YOLO11 classifier on dataset: farm_dataset
📊 Model: yolo11l-cls.pt (will be converted to classification), Epochs: 200, Batch size: 16
🖼️  Image size: 640, Device: 0
📁 Using classification dataset structure (train/val/test folders with class subdirectories)
📈 TensorBoard logging: ENABLED
   💡 After training starts, run the TensorBoard monitoring cell below to view real-time metrics!
✅ Classification dataset found at: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset
🚀 Starting classification training with live progress...
🔄 Detection model will be automatically converted to classification model
📊 You should see epoch progress, loss values, and accuracy metrics below:
📈 TensorBoard logs will be saved to runs/train/farm_dataset_classifier/
📊 Classification dataset: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset
Loading model: yolo11l-cls.pt
📊 Classification dataset: /home/vgreens/SSD_drive_1.8T/strawberry_cl

100%|██████████| 27.2M/27.2M [00:01<00:00, 20.4MB/s]
100%|██████████| 27.2M/27.2M [00:01<00:00, 20.4MB/s]


📈 TensorBoard logging enabled - logs will be saved automatically
🚀 Starting YOLO11 classification training...
New https://pypi.org/project/ultralytics/8.3.168 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.166 🚀 Python-3.10.12 torch-2.7.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070, 11903MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf

train: Scanning /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/train... 3304 images, 1 corrupt: 100%|██████████| 3305/3305 [00:00<00:00, 10681.39it/s]
val: Scanning /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/val... 913 images, 2 corrupt: 100%|██████████| 913/913 [00:00<?, ?it/s]
train: Scanning /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/train... 3304 images, 1 corrupt: 100%|██████████| 3305/3305 [00:00<00:00, 10681.39it/s]
val: Scanning /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/val... 913 images, 2 corrupt: 100%|██████████| 913/913 [00:00<?, ?it/s]


train: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/train/Flower/61a20fb1-IMG_0428_crop_006.jpg: ignoring corrupt image/label: image size (5, 9) <10 pixels
train: New cache created: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1483.8±520.9 MB/s, size: 22.9 KB)
val: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/val/Flower_Fruit/9035b80b-IMG_9281_crop_014.jpg: ignoring corrupt image/label: image size (9, 158) <10 pixels
val: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/data/classification/farm_dataset/val/Red_Fruit/ffb0effa-IMG_8176_crop_041.jpg: ignoring corrupt image/label: image size (1, 5) <10 pixels
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: SGD(lr=0.01, momentum=0.9) with parameter grou

      1/200      6.63G      1.262          8        640: 100%|██████████| 207/207 [00:38<00:00,  5.44it/s]
      1/200      6.63G      1.262          8        640: 100%|██████████| 207/207 [00:38<00:00,  5.44it/s]
               classes   top1_acc   top5_acc: 100%|██████████| 29/29 [00:03<00:00,  8.85it/s]


                   all      0.634      0.986

      Epoch    GPU_mem       loss  Instances       Size

      Epoch    GPU_mem       loss  Instances       Size


      2/200      6.42G     0.9602         16        640:  71%|███████   | 147/207 [00:23<00:09,  6.42it/s]

## 📈 TensorBoard Monitoring

Monitor your training progress in real-time with TensorBoard. YOLO11 automatically generates TensorBoard logs when TensorBoard is available in your environment.

**How it works:**
- When `ENABLE_TENSORBOARD = True`, the training script ensures TensorBoard is installed
- YOLO11 automatically detects TensorBoard and creates event logs during training
- TensorBoard logs are saved alongside other training outputs in `runs/train/{experiment_name}/`

**Features:**
- Real-time loss and metric tracking
- Learning rate schedules  
- Model architecture visualization
- Training/validation comparisons
- Experiment comparison across multiple runs

In [11]:
# TensorBoard Configuration
TENSORBOARD_LOGDIR = "runs/train"  # Directory containing your training experiments
TENSORBOARD_HOST = "localhost"     # Host to bind TensorBoard server
TENSORBOARD_PORT = 6006           # Port for TensorBoard server

# Check if TensorBoard is installed
try:
    import tensorboard
    print("✅ TensorBoard is installed")
except ImportError:
    print("❌ TensorBoard not found. Installing...")
    subprocess.run([sys.executable, "-m", "pip", "install", "tensorboard"], check=True)
    print("✅ TensorBoard installed successfully!")

# Check for existing TensorBoard logs
logdir_path = PROJECT_ROOT / TENSORBOARD_LOGDIR
print(f"\n📁 Checking for TensorBoard logs in: {logdir_path}")

if logdir_path.exists():
    # Look for experiment directories
    experiments = [d for d in logdir_path.iterdir() if d.is_dir()]
    if experiments:
        print(f"📊 Found {len(experiments)} experiment(s):")
        for exp in sorted(experiments):
            # Check for TensorBoard event files (YOLO11 creates these automatically)
            tb_files = list(exp.glob("events.out.tfevents*"))
            if tb_files:
                print(f"  ✅ {exp.name} (with TensorBoard logs - {len(tb_files)} files)")
            else:
                print(f"  ⚠️  {exp.name} (no TensorBoard logs found)")
                
            # Also check for other YOLO outputs
            weight_files = list(exp.glob("weights/*.pt"))
            csv_files = list(exp.glob("*.csv"))
            if weight_files:
                print(f"      📦 {len(weight_files)} model weights")
            if csv_files:
                print(f"      📊 {len(csv_files)} results files")
    else:
        print("⚠️  No experiment directories found")
else:
    print("⚠️  Logs directory doesn't exist yet")

print(f"\n📈 YOLO11 + TensorBoard Integration:")
print("   • YOLO11 automatically generates TensorBoard logs when TensorBoard is available")
print("   • Training & validation loss curves")
print("   • Precision, recall, and mAP metrics over time")
print("   • Learning rate schedules")
print("   • Model architecture visualization")
print("   • Image samples and predictions")
print("   • Easy experiment comparison across runs")

print(f"\n🚀 To launch TensorBoard:")
print(f"   1. Make sure you have run training with ENABLE_TENSORBOARD=True")
print(f"   2. Run the command below to start TensorBoard:")
print(f"      tensorboard --logdir={TENSORBOARD_LOGDIR} --host={TENSORBOARD_HOST} --port={TENSORBOARD_PORT}")
print(f"   3. Open your browser to: http://{TENSORBOARD_HOST}:{TENSORBOARD_PORT}")

# Optional: Launch TensorBoard automatically
LAUNCH_TENSORBOARD = False  # Set to True to auto-launch TensorBoard

if LAUNCH_TENSORBOARD and logdir_path.exists():
    # Check if any experiments have TensorBoard logs
    has_tb_logs = False
    if logdir_path.exists():
        for exp_dir in logdir_path.iterdir():
            if exp_dir.is_dir() and list(exp_dir.glob("events.out.tfevents*")):
                has_tb_logs = True
                break
    
    if has_tb_logs:
        print(f"\n🚀 Launching TensorBoard on http://{TENSORBOARD_HOST}:{TENSORBOARD_PORT}...")
        
        # Launch TensorBoard as a background process
        tensorboard_cmd = [
            sys.executable, "-m", "tensorboard.main",
            "--logdir", str(logdir_path),
            "--host", TENSORBOARD_HOST,
            "--port", str(TENSORBOARD_PORT)
        ]
        
        try:
            # Start TensorBoard in background
            tensorboard_process = subprocess.Popen(
                tensorboard_cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE
            )
            print(f"✅ TensorBoard started! PID: {tensorboard_process.pid}")
            print(f"🌐 Access TensorBoard at: http://{TENSORBOARD_HOST}:{TENSORBOARD_PORT}")
            print("💡 To stop TensorBoard, restart this notebook kernel or kill the process")
            
        except Exception as e:
            print(f"❌ Failed to launch TensorBoard: {e}")
            print("💡 You can manually run: tensorboard --logdir=runs/train")
    else:
        print("⚠️  No TensorBoard logs found in experiments")
        print("💡 Run a training session first with ENABLE_TENSORBOARD=True")
elif LAUNCH_TENSORBOARD:
    print("⚠️  Cannot launch TensorBoard: no log directory found")
    print("💡 Run a training session first with ENABLE_TENSORBOARD=True")

✅ TensorBoard is installed

📁 Checking for TensorBoard logs in: /home/vgreens/SSD_drive_1.8T/strawberry_classifier/runs/train
📊 Found 3 experiment(s):
  ⚠️  farm_dataset_640_l_pretrained (no TensorBoard logs found)
      📦 8 model weights
      📊 1 results files
  ⚠️  open_source_dataset_640_l (no TensorBoard logs found)
      📦 17 model weights
      📊 1 results files
  ⚠️  strawberry_classifier_640_l (no TensorBoard logs found)
      📦 15 model weights
      📊 1 results files

📈 YOLO11 + TensorBoard Integration:
   • YOLO11 automatically generates TensorBoard logs when TensorBoard is available
   • Training & validation loss curves
   • Precision, recall, and mAP metrics over time
   • Learning rate schedules
   • Model architecture visualization
   • Image samples and predictions
   • Easy experiment comparison across runs

🚀 To launch TensorBoard:
   1. Make sure you have run training with ENABLE_TENSORBOARD=True
   2. Run the command below to start TensorBoard:
      tensorboard -

## 🔍 Model Validation

Validate your trained model on the test dataset to evaluate performance.

In [ ]:
# Configuration for model validation
MODEL_PATH = "runs/detect/train/weights/best.pt"  # Path to your trained model
DATASET_NAME = "farm_dataset"  # Change this to your dataset name
DEVICE = "auto"  # auto, cpu, or cuda:0

# Execute model validation
print(f"🔍 Validating model: {MODEL_PATH}")
print(f"📊 Dataset: {DATASET_NAME}, Device: {DEVICE}")
print("=" * 60)

try:
    cmd = [
        sys.executable, 
        str(SCRIPTS_DIR / "validate.py"),
        "--model", MODEL_PATH,
        "--dataset", DATASET_NAME,
        "--device", DEVICE
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    
    if result.returncode == 0:
        print("✅ Model validation completed successfully!")
        print(result.stdout)
    else:
        print("❌ Error during model validation:")
        print(result.stderr)
        
except Exception as e:
    print(f"❌ Failed to run model validation: {e}")

## 🎯 Model Prediction

Use your trained model to make predictions on new images.

In [ ]:
# Configuration for model prediction
MODEL_PATH = "runs/detect/train/weights/best.pt"  # Path to your trained model
SOURCE = "data/datasets/farm_dataset/images/"  # Path to image(s) or directory
CONFIDENCE = 0.25  # Confidence threshold
SAVE_DIR = "predictions"  # Directory to save predictions
DEVICE = "auto"  # auto, cpu, or cuda:0

# Execute model prediction
print(f"🎯 Making predictions with model: {MODEL_PATH}")
print(f"📁 Source: {SOURCE}, Confidence: {CONFIDENCE}")
print(f"💾 Save directory: {SAVE_DIR}, Device: {DEVICE}")
print("=" * 60)

try:
    cmd = [
        sys.executable, 
        str(SCRIPTS_DIR / "predict.py"),
        "--model", MODEL_PATH,
        "--source", SOURCE,
        "--conf", str(CONFIDENCE),
        "--save-dir", SAVE_DIR,
        "--device", DEVICE
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=PROJECT_ROOT)
    
    if result.returncode == 0:
        print("✅ Model prediction completed successfully!")
        print(result.stdout)
    else:
        print("❌ Error during model prediction:")
        print(result.stderr)
        
except Exception as e:
    print(f"❌ Failed to run model prediction: {e}")